# 小鼠与猕猴：原选基因的共同刻度表达展示

只展示 `mouse` 和 `macaque`，不展示大鼠。沿用原 notebook 已导出的选基因表，将各簇对的基因按首次出现的顺序去重；不重新选基因或进行差异检验，也不沿用原来的大鼠类别标签和簇配对。

输入使用现有 `three_species_processed.rds`，读取后立即筛选小鼠和猕猴，保留其全部细胞及已有 cluster 标签。该对象的 RNA counts 已统一为大写共同基因；不会重新聚类、整合或改变原始 RDS。

绘图时先将两物种的表达合并为一张表，再用一次 `facet_grid` 绘图：每行一个基因，每列一个物种，各列展示本物种全部簇，**同一个基因在两物种间共享完全相同的表达量纵轴**，不同基因可使用不同纵轴。使用 RNA `log1p(CP10K)`，保留零表达；不使用 integrated 或分别按物种计算的 Z-score。

选择 **mb4.5.2** R 内核按单元运行。参数、合并后的 `plot_obj`、`plot_data`、`gene_scale`、`violin_plot` 均可直接查看；图和核对表写入独立目录。

In [ ]:
#%% 参数与依赖
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(ggplot2)
})
root <- '/mnt/90-connectome/Personal/CQW/midbrain'
processed_path <- file.path(root, 'analysis/three_species_DA_20261109/three_species_processed.rds')
selected_genes_path <- file.path(root, 'analysis/cluster_species_deg_heatmap_20261101_new/tables/violin_selected_genes.csv')
out <- file.path(root, 'analysis/mouse_macaque_shared_scale_20261110')
species_order <- c('mouse', 'macaque')
species_labels <- c(mouse = 'Mouse', macaque = 'Macaque')
species_colors <- c(mouse = '#8AB16D', macaque = '#DD806B')
plot_font <- 'sans'
violin_width <- 12
gene_row_height <- 0.45
dir.create(out, recursive = TRUE, showWarnings = FALSE)

In [ ]:
#%% 仅保留小鼠与猕猴，合并对象中保留原来的 cluster 标签
processed <- readRDS(processed_path)
stopifnot(all(c('species', 'cluster') %in% names(processed[[]])))
plot_cells <- colnames(processed)[processed$species %in% species_order]
plot_obj <- subset(processed, cells = plot_cells)
rm(processed)
invisible(gc())
DefaultAssay(plot_obj) <- 'RNA'
plot_obj@misc$species_comparison <- NULL
plot_obj$species <- factor(as.character(plot_obj$species), levels = species_order)
stopifnot(setequal(as.character(unique(plot_obj$species)), species_order),
          !anyNA(plot_obj$cluster), all(nzchar(as.character(plot_obj$cluster))))
print(table(plot_obj$species, plot_obj$cluster))

In [ ]:
#%% 只沿用原来的基因名单，不将原来的差异类别解释为小鼠的差异结果
original_selection <- read.csv(selected_genes_path, check.names = FALSE)
requested_genes <- unique(as.character(original_selection$gene))
stopifnot(length(requested_genes) > 0L, !anyNA(requested_genes), all(nzchar(requested_genes)))
gene_presence <- data.frame(gene = requested_genes,
  present = requested_genes %in% rownames(plot_obj[['RNA']]))
missing_genes <- gene_presence$gene[!gene_presence$present]
if (length(missing_genes)) {
  warning('当前共同基因对象中缺少以下原选基因：', paste(missing_genes, collapse = ', '))
}
gene_order <- gene_presence$gene[gene_presence$present]
stopifnot(length(gene_order) > 0L)
cat('原选基因：', length(requested_genes), '；实际展示：', length(gene_order), '\n')

In [ ]:
#%% 在合并后的对象上统一计算 RNA log1p(CP10K)，再构建同一张绘图表
raw_counts <- LayerData(plot_obj, assay = 'RNA', layer = 'counts')
cell_library <- Matrix::colSums(raw_counts)
stopifnot(all(is.finite(cell_library)), all(cell_library > 0))
plot_obj <- NormalizeData(plot_obj, assay = 'RNA', normalization.method = 'LogNormalize',
  scale.factor = 10000, verbose = FALSE)
expression <- as.matrix(LayerData(plot_obj, assay = 'RNA', layer = 'data')[gene_order, , drop = FALSE])
plot_meta <- plot_obj[[]][colnames(expression), , drop = FALSE]
cluster_levels <- unique(as.character(plot_meta$cluster))
cluster_numbers <- as.integer(gsub('[^0-9]', '', cluster_levels))
stopifnot(!anyNA(cluster_numbers))
cluster_levels <- cluster_levels[order(cluster_numbers, cluster_levels)]
plot_data <- data.frame(
  gene = factor(rep(gene_order, ncol(expression)), levels = gene_order),
  cell = rep(colnames(expression), each = length(gene_order)),
  species = factor(rep(plot_meta$species, each = length(gene_order)), levels = species_order),
  cluster = factor(rep(plot_meta$cluster, each = length(gene_order)), levels = cluster_levels),
  expression = as.vector(expression))
stopifnot(!anyNA(plot_data), all(is.finite(plot_data$expression)),
          all(plot_data$expression >= 0),
          setequal(as.character(unique(plot_data$species)), species_order))
# 每个基因的展示范围取两物种合并后的范围，供核对共享刻度。
gene_scale <- data.frame(gene = gene_order, minimum = 0,
  maximum = pmax(apply(expression, 1, max), 0.1), row.names = NULL)

In [ ]:
#%% 常数表达组用短横线表示，其余组绘制小提琴；不删除零表达细胞
groups <- split(seq_len(nrow(plot_data)),
  interaction(plot_data$gene, plot_data$species, plot_data$cluster, drop = TRUE))
variable <- vapply(groups, function(rows) length(rows) >= 2L &&
  any(plot_data$expression[rows] != plot_data$expression[rows[1]]), logical(1))
density_rows <- unlist(groups[variable], use.names = FALSE)
constant_rows <- vapply(groups[!variable], function(rows) rows[1], integer(1))

In [ ]:
#%% 一次绘图：facet_grid 的同行面板共用 y 轴，按物种列分别保留各自的簇
violin_plot <- ggplot(plot_data, aes(cluster, expression, fill = species)) +
  geom_violin(data = plot_data[density_rows, , drop = FALSE],
    scale = 'width', trim = TRUE, linewidth = 0.2, width = 0.85) +
  geom_point(data = plot_data[constant_rows, , drop = FALSE], shape = 95, size = 3) +
  facet_grid(rows = vars(gene), cols = vars(species), scales = 'free',
    space = 'free_x', switch = 'y', labeller = labeller(species = species_labels)) +
  scale_fill_manual(values = species_colors, breaks = species_order, drop = TRUE) +
  scale_y_continuous(limits = function(z) c(0, max(z[2], 0.1)),
    expand = expansion(mult = c(0, 0.06))) +
  labs(x = 'Cluster', y = 'RNA expression log1p(CP10K)') +
  theme_classic(base_size = 9, base_family = plot_font) +
  theme(legend.position = 'none', strip.background = element_blank(),
    strip.placement = 'outside', strip.text.x = element_text(face = 'bold', size = 12),
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = 8),
    panel.spacing.x = grid::unit(0.8, 'lines'), panel.spacing.y = grid::unit(0.1, 'lines'),
    axis.text.x = element_text(angle = 45, hjust = 1), axis.text.y = element_text(size = 6))
violin_height <- max(5, length(gene_order) * gene_row_height + 2)
options(repr.plot.width = violin_width, repr.plot.height = violin_height)
print(violin_plot)

In [ ]:
#%% 保存图及核对信息；原来的 DEG 结果、RDS 和其他 notebook 不变
ggsave(file.path(out, 'mouse_macaque_shared_scale_violin.pdf'), violin_plot,
  width = violin_width, height = violin_height, units = 'in',
  device = grDevices::cairo_pdf, family = plot_font, limitsize = FALSE)
ggsave(file.path(out, 'mouse_macaque_shared_scale_violin.png'), violin_plot,
  width = violin_width, height = violin_height, units = 'in', dpi = 180,
  bg = 'white', limitsize = FALSE)
write.csv(gene_presence, file.path(out, 'original_gene_presence.csv'), row.names = FALSE)
write.csv(gene_scale, file.path(out, 'shared_gene_expression_ranges.csv'), row.names = FALSE)
cell_counts <- as.data.frame(table(species = plot_meta$species, cluster = plot_meta$cluster))
cell_counts <- cell_counts[cell_counts$Freq > 0, ]
write.csv(cell_counts, file.path(out, 'species_cluster_cell_counts.csv'), row.names = FALSE)
cat('结果已保存至：', out, '\n')